# SensorFusion-HAR: Lightweight Real-Time Human Activity Recognition


## Architecture Overview

4-stage pipeline: Echo State Network (reservoir) -> DS-Conv1D -> Patch Micro-Attention -> Binary Head

Target: ~22KB model, <50ms inference, 94%+ accuracy on UCI-HAR


In [ ]:
import os
import json
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from model import SensorFusionHAR
from model.dataset import UCIHARDataset
from model.reservoir import EchoStateNetwork
from model.dsconv import DSConvEncoder
from model.attention import PatchMicroAttention
from model.binary_head import BinaryClassifier

plt.style.use('dark_background')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Dataset


In [ ]:
DATA_DIR = "data/UCI HAR Dataset"

if not os.path.isdir(DATA_DIR):
    UCIHARDataset.download("data")

train_ds = UCIHARDataset(DATA_DIR, split="train")
test_ds = UCIHARDataset(DATA_DIR, split="test")

print(f"Train: {train_ds.X.shape}, Test: {test_ds.X.shape}")
print(f"Train labels: {train_ds.y.shape}, Test labels: {test_ds.y.shape}")

LABELS = ["Walking", "Upstairs", "Downstairs", "Sitting", "Standing", "Laying"]
COLORS = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#F44336", "#00BCD4"]

for i in range(6):
    count = (train_ds.y == i).sum().item()
    print(f"  {LABELS[i]}: {count}")

means, stds = UCIHARDataset.get_normalization_stats(DATA_DIR)
print(f"\nChannel means: {[round(m, 4) for m in means]}")
print(f"Channel stds:  {[round(s, 4) for s in stds]}")

mean_t = torch.tensor(means, dtype=torch.float32)
std_t = torch.tensor(stds, dtype=torch.float32)
train_ds.X = (train_ds.X - mean_t) / std_t
test_ds.X = (test_ds.X - mean_t) / std_t

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
channel_names = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
sample_activities = [0, 1, 3, 5]

for idx, act in enumerate(sample_activities):
    ax = axes[idx // 2][idx % 2]
    mask = train_ds.y == act
    sample_idx = torch.where(mask)[0][0].item()
    window = train_ds.X[sample_idx].numpy()
    for ch in range(6):
        ax.plot(window[:, ch], linewidth=0.8, alpha=0.8, label=channel_names[ch])
    ax.set_title(LABELS[act], color=COLORS[act], fontsize=14, fontweight="bold")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Normalized Value")
    if idx == 0:
        ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.savefig("sample_windows.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Architecture


In [ ]:
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)

print(model)
print()

for name, module in [("Reservoir (ESN)", model.reservoir), ("DS-Conv Encoder", model.dsconv), ("Patch Attention", model.attention), ("Binary Classifier", model.classifier)]:
    params = sum(p.numel() for p in module.parameters() if p.requires_grad)
    buffers = sum(b.numel() for b in module.buffers())
    print(f"{name:25s} | Trainable: {params:>8,} | Buffers: {buffers:>6,}")

print(f"\nTotal trainable: {model.count_parameters():,}")
print(f"Model size (FP32): {model.model_size_kb():.2f} KB")
print(f"Model size (INT8): {model.quantized_size_kb():.2f} KB")

dummy = torch.randn(1, 128, 6).to(device)
out = model(dummy)
print(f"\nForward pass: {dummy.shape} -> {out.shape}")

## Training


In [ ]:
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.001

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

os.makedirs("checkpoints", exist_ok=True)
best_acc = 0.0
history = {"train_loss": [], "test_loss": [], "test_acc": [], "test_f1": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    train_loss = epoch_loss / len(train_ds)

    model.eval()
    test_loss_total = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            test_loss_total += criterion(out, y).item() * X.size(0)
            all_preds.append(out.argmax(1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    test_loss = test_loss_total / len(test_ds)
    test_acc = (all_preds == all_labels).mean()
    test_f1 = f1_score(all_labels, all_preds, average="macro")
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    history["test_f1"].append(test_f1)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "accuracy": test_acc, "f1_macro": test_f1, "normalization_stats": {"means": means, "stds": stds}}, "checkpoints/best_model.pt")

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Acc: {test_acc*100:.2f}% | F1: {test_f1:.4f}")

print(f"\nBest accuracy: {best_acc*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
epochs_range = range(1, EPOCHS + 1)
best_epoch = np.argmax(history["test_acc"]) + 1

axes[0, 0].plot(epochs_range, history["train_loss"], color="#FF6B6B", linewidth=1.2)
axes[0, 0].set_title("Train Loss")
axes[0, 0].axvline(best_epoch, color="white", linestyle="--", alpha=0.5)

axes[0, 1].plot(epochs_range, history["test_loss"], color="#4ECDC4", linewidth=1.2)
axes[0, 1].set_title("Test Loss")
axes[0, 1].axvline(best_epoch, color="white", linestyle="--", alpha=0.5)

axes[1, 0].plot(epochs_range, [a * 100 for a in history["test_acc"]], color="#45B7D1", linewidth=1.2)
axes[1, 0].set_title("Test Accuracy (%)")
axes[1, 0].axvline(best_epoch, color="white", linestyle="--", alpha=0.5)

axes[1, 1].plot(epochs_range, history["test_f1"], color="#96CEB4", linewidth=1.2)
axes[1, 1].set_title("Test F1 Macro")
axes[1, 1].axvline(best_epoch, color="white", linestyle="--", alpha=0.5)

for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Evaluation


In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        all_preds.append(out.argmax(1).cpu().numpy())
        all_labels.append(y.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

acc = (all_preds == all_labels).mean()
f1_macro = f1_score(all_labels, all_preds, average="macro")
f1_per = f1_score(all_labels, all_preds, average=None)

print(f"Accuracy: {acc*100:.2f}%")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"\nBest epoch: {checkpoint['epoch']}")
print(f"\n{classification_report(all_labels, all_preds, target_names=LABELS)}")

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="YlOrRd")
ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

for i in range(6):
    for j in range(6):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=12)

plt.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(6), f1_per, color=COLORS, height=0.6)
ax.set_yticks(range(6))
ax.set_yticklabels(LABELS)
ax.set_xlabel("F1 Score")
ax.set_title("Per-Class F1 Score")
ax.set_xlim(0, 1.05)

for bar, val in zip(bars, f1_per):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontsize=11)

plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation Study

In [ ]:
class NoReservoirModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj = nn.Linear(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.classifier = BinaryClassifier(in_features=32, num_classes=6)
    def forward(self, x):
        x = self.input_proj(x)
        x = x.transpose(1, 2)
        x = self.dsconv(x)
        x = self.attention(x)
        return self.classifier(x)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoAttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = BinaryClassifier(in_features=48, num_classes=6)
    def forward(self, x):
        x = self.reservoir(x)
        x = x.transpose(1, 2)
        x = self.dsconv(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoBinaryHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.dsconv = DSConvEncoder(in_channels=32)
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.bn = nn.BatchNorm1d(32)
        self.head = nn.Linear(32, 6)
    def forward(self, x):
        x = self.reservoir(x)
        x = x.transpose(1, 2)
        x = self.dsconv(x)
        x = self.attention(x)
        return self.head(self.bn(x))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class NoDSConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.reservoir = EchoStateNetwork(6, 32)
        self.conv = nn.Sequential(
            nn.Conv1d(32, 48, kernel_size=5, stride=1, padding=2), nn.BatchNorm1d(48), nn.ReLU(inplace=True),
            nn.Conv1d(48, 48, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(48), nn.ReLU(inplace=True),
            nn.Conv1d(48, 48, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(48), nn.ReLU(inplace=True),
        )
        self.attention = PatchMicroAttention(in_channels=48, seq_len=32, d_model=32, ff_dim=48)
        self.classifier = BinaryClassifier(in_features=32, num_classes=6)
    def forward(self, x):
        x = self.reservoir(x)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = self.attention(x)
        return self.classifier(x)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

ABLATION_EPOCHS = 50
ablation_variants = [
    ("Full Model", SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6)),
    ("No Reservoir", NoReservoirModel()),
    ("No Attention", NoAttentionModel()),
    ("No Binary Head", NoBinaryHeadModel()),
    ("No DS-Conv", NoDSConvModel()),
]

ablation_results = []
for vname, vmodel in ablation_variants:
    vmodel = vmodel.to(device)
    vparams = vmodel.count_parameters()
    print(f"\n--- {vname} ({vparams:,} params) ---")
    vopt = torch.optim.AdamW(vmodel.parameters(), lr=LR, weight_decay=1e-4)
    vsched = torch.optim.lr_scheduler.CosineAnnealingLR(vopt, T_max=ABLATION_EPOCHS)
    best_vacc = 0.0
    for ep in range(1, ABLATION_EPOCHS + 1):
        vmodel.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            vopt.zero_grad()
            nn.CrossEntropyLoss()(vmodel(X), y).backward()
            vopt.step()
        vsched.step()
        vmodel.eval()
        vp, vl = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                vp.append(vmodel(X).argmax(1).cpu().numpy())
                vl.append(y.cpu().numpy())
        vp, vl = np.concatenate(vp), np.concatenate(vl)
        vacc = (vp == vl).mean()
        if vacc > best_vacc:
            best_vacc = vacc
        if ep % 10 == 0:
            print(f"  Epoch {ep}/{ABLATION_EPOCHS}: acc={vacc*100:.2f}%")
    vf1 = f1_score(vl, vp, average="macro")
    ablation_results.append((vname, vparams, best_vacc, vf1))
    print(f"  Best: {best_vacc*100:.2f}% | F1: {vf1:.4f}")

print(f"\n{'='*60}")
print(f"{'Variant':<20s}{'Params':>10s}{'Accuracy':>12s}{'F1 Macro':>12s}")
print(f"{'-'*54}")
for vname, vparams, vacc, vf1 in ablation_results:
    print(f"{vname:<20s}{vparams:>10,}{vacc*100:>11.2f}%{vf1:>12.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
names = [r[0] for r in ablation_results]
accs = [r[2] * 100 for r in ablation_results]
f1s = [r[3] * 100 for r in ablation_results]
y_pos = np.arange(len(names))
bar_height = 0.35

bars1 = ax.barh(y_pos - bar_height/2, accs, bar_height, label="Accuracy", color="#45B7D1", alpha=0.9)
bars2 = ax.barh(y_pos + bar_height/2, f1s, bar_height, label="F1 Macro", color="#96CEB4", alpha=0.9)

for bar in bars1:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}%", va="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}%", va="center", fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel("Score (%)")
ax.set_title("Ablation Study Results")
ax.legend(loc="lower right")
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig("ablation_results.png", dpi=150, bbox_inches="tight")
plt.show()

## Inference Benchmark


In [ ]:
model.eval()
sample = test_ds.X[0:1].to(device)

for _ in range(50):
    with torch.no_grad():
        model(sample)

latencies = []
for _ in range(1000):
    t0 = time.perf_counter()
    with torch.no_grad():
        model(sample)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)

print(f"Inference Latency (1000 runs):")
print(f"  Mean:  {latencies.mean():.3f} ms")
print(f"  Std:   {latencies.std():.3f} ms")
print(f"  P95:   {np.percentile(latencies, 95):.3f} ms")
print(f"  P99:   {np.percentile(latencies, 99):.3f} ms")
print(f"  Min:   {latencies.min():.3f} ms")
print(f"  Max:   {latencies.max():.3f} ms")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(latencies, bins=50, color="#4ECDC4", edgecolor="none", alpha=0.8)
ax.axvline(latencies.mean(), color="#FF6B6B", linestyle="--", linewidth=2, label=f"Mean: {latencies.mean():.2f}ms")
ax.axvline(np.percentile(latencies, 95), color="#FFE66D", linestyle="--", linewidth=2, label=f"P95: {np.percentile(latencies, 95):.2f}ms")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Inference Latency Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Export


In [ ]:
export_path = "checkpoints/best_model.pt"
size_bytes = os.path.getsize(export_path)
print(f"Model file: {export_path}")
print(f"File size: {size_bytes / 1024:.2f} KB")
print(f"Trainable parameters: {model.count_parameters():,}")
print(f"Estimated FP32 size: {model.model_size_kb():.2f} KB")
print(f"Estimated INT8 size: {model.quantized_size_kb():.2f} KB")

verify = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
state = torch.load(export_path, map_location=device, weights_only=False)
verify.load_state_dict(state["model_state_dict"])
verify.eval()

with torch.no_grad():
    out1 = model(sample)
    out2 = verify(sample)

match = torch.allclose(out1, out2, atol=1e-6)
print(f"\nVerification: {'PASSED' if match else 'FAILED'}")